# Book Recommender System

In [1]:
import numpy as np
import pandas as pd

In [2]:
books = pd.read_csv('data/books.csv', encoding='latin-1')

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_12652\1563079984.py:1: DtypeWarning: Columns (0: Year-Of-Publication) have mixed types. Specify dtype option on import or set low_memory=False.
  books = pd.read_csv('data/books.csv', encoding='latin-1')


In [3]:
books.head()

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...


In [4]:
books.shape

(271360, 8)

In [5]:
users = pd.read_csv('data/users.csv')

In [6]:
users.head(2)

,User-ID,Location,Age
0,1,"nyc, new york, usa",NaN
1,2,"stockton, california, usa",18.0


In [7]:
users.rename(columns={'User-Id': 'user_id', 'Location': 'location', 'Age': 'age'}, inplace=True)

In [8]:
ratings = pd.read_csv('data/ratings.csv')

In [9]:
ratings.head(2)

,User-ID,ISBN,Book-Rating
0,276725,034545104X,0
1,276726,0155061224,5


In [10]:
ratings.rename(columns={'User-ID': 'user_id', 'Book-Rating': 'book_rating'}, inplace=True)

In [11]:
ratings.columns

Index(['user_id', 'ISBN', 'book_rating'], dtype='str')

In [12]:
print(books.shape)
print(users.shape)
print(ratings.shape)

(271360, 8)
(278858, 3)
(1149780, 3)


In [13]:
books.isnull().sum()

ISBN                   0
Book-Title             0
Book-Author            2
Year-Of-Publication    0
Publisher              2
Image-URL-S            0
Image-URL-M            0
Image-URL-L            3
dtype: int64

In [14]:
users.isnull().sum()

User-ID          0
location         0
age         110762
dtype: int64

In [15]:
ratings.isnull().sum()

user_id        0
ISBN           0
book_rating    0
dtype: int64

In [16]:
books.duplicated().sum()

np.int64(0)

In [17]:
ratings.duplicated().sum()

np.int64(0)

In [18]:
users.duplicated().sum()

np.int64(0)

## Let's find top 50 popular books based on max average ratings which are rated by more than 250 users

In [19]:
ratings_with_name=ratings.merge(books, on='ISBN')

In [20]:
ratings_with_name.head(2)

,user_id,ISBN,book_rating,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,276725,034545104X,0,Flesh Tones: A Novel,M. J. Rose,2002,Ballantine Books,http://images.amazon.com/images/P/034545104X.0...,http://images.amazon.com/images/P/034545104X.0...,http://images.amazon.com/images/P/034545104X.0...
1,276726,0155061224,5,Rites of Passage,Judith Rae,2001,Heinle,http://images.amazon.com/images/P/0155061224.0...,http://images.amazon.com/images/P/0155061224.0...,http://images.amazon.com/images/P/0155061224.0...


In [21]:
num_rating_df=ratings_with_name.groupby('Book-Title')['book_rating'].count().sort_values(ascending=False).reset_index()

In [22]:
num_rating_df.head(2)

,Book-Title,book_rating
0,Wild Animus,2502
1,The Lovely Bones: A Novel,1295


In [23]:
num_rating_df.rename(columns={'book_rating': 'num_ratings'}, inplace=True)

In [24]:
avg_rating_df=ratings_with_name.groupby('Book-Title')['book_rating'].mean().sort_values(ascending=False).reset_index()
avg_rating_df.rename(columns={'book_rating': 'avg_rating'}, inplace=True)

In [25]:
avg_rating_df.head(2)

,Book-Title,avg_rating
0,Timelock: How Life Got So Hectic and What You ...,10.0
1,Timelines of World History,10.0


In [26]:
popular_df=num_rating_df.merge(avg_rating_df, on='Book-Title')
popular_df.head(2)

,Book-Title,num_ratings,avg_rating
0,Wild Animus,2502,1.019584
1,The Lovely Bones: A Novel,1295,4.468726


In [27]:
popular_df=popular_df[popular_df['num_ratings']>250].sort_values('avg_rating', ascending=False).head(50)

In [28]:
popular_df.shape

(50, 3)

In [29]:
books.shape

(271360, 8)

In [30]:
books.columns

Index(['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher',
       'Image-URL-S', 'Image-URL-M', 'Image-URL-L'],
      dtype='str')

In [31]:
popular_df=popular_df.merge(books, on='Book-Title').drop_duplicates('Book-Title')

In [32]:
popular_df.shape

(50, 10)

In [33]:
popular_df = popular_df[['Book-Title', 'Book-Author', 'Image-URL-M', 'num_ratings', 'avg_rating']]

In [34]:
# top 50 popular books
popular_df.shape

(50, 5)

In [35]:
#
import pickle
pickle.dump(popular_df, open('popular_books.pkl', 'wb'))

### Finding each users with their total ratings on books

In [36]:
ratings['user_id'].value_counts()

user_id
11676     13602
198711     7550
153662     6109
98391      5891
35859      5850
          ...  
276697        1
276706        1
276709        1
276721        1
276723        1
Name: count, Length: 105283, dtype: int64

value_counts() returns a series with index and value
considering only those users who rated more than 200 books

In [37]:
ratings['user_id'].value_counts()>200

user_id
11676      True
198711     True
153662     True
98391      True
35859      True
          ...  
276697    False
276706    False
276709    False
276721    False
276723    False
Name: count, Length: 105283, dtype: bool

Storing the returned series in variable x with user_id as index of the series and boolean value(true,false) as values

In [38]:
x = ratings['user_id'].value_counts()>200

In [39]:
x.shape

(105283,)

In [40]:
# keeping only the users who have rated more than 200 books
y = x[x].index

In [41]:
y.shape

(899,)

### Only 899 users are our real users. Because we will consider their ratings as valid ratings

In [42]:
# filtering the ratings dataframe to only include users who have rated more than 200 books
ratings = ratings[ratings['user_id'].isin(y)]

In [43]:
ratings.shape

(526356, 3)

In [44]:
# merging the ratings and books dataframes on the ISBN column
ratings_with_books = ratings.merge(books, on='ISBN')

In [45]:
ratings_with_books.shape

(487671, 10)

#### ratings_with_books has less rows than ratings because some of the ISBNs in ratings do not have a corresponding entry in books

In [46]:
ratings_with_books.head()

,user_id,ISBN,book_rating,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,277427,002542730X,10,Politically Correct Bedtime Stories: Modern Ta...,James Finn Garner,1994,John Wiley &amp; Sons Inc,http://images.amazon.com/images/P/002542730X.0...,http://images.amazon.com/images/P/002542730X.0...,http://images.amazon.com/images/P/002542730X.0...
1,277427,0026217457,0,Vegetarian Times Complete Cookbook,Lucy Moll,1995,John Wiley &amp; Sons,http://images.amazon.com/images/P/0026217457.0...,http://images.amazon.com/images/P/0026217457.0...,http://images.amazon.com/images/P/0026217457.0...
2,277427,003008685X,8,Pioneers,James Fenimore Cooper,1974,Thomson Learning,http://images.amazon.com/images/P/003008685X.0...,http://images.amazon.com/images/P/003008685X.0...,http://images.amazon.com/images/P/003008685X.0...
3,277427,0030615321,0,"Ask for May, Settle for June (A Doonesbury book)",G. B. Trudeau,1982,Henry Holt &amp; Co,http://images.amazon.com/images/P/0030615321.0...,http://images.amazon.com/images/P/0030615321.0...,http://images.amazon.com/images/P/0030615321.0...
4,277427,0060002050,0,On a Wicked Dawn (Cynster Novels),Stephanie Laurens,2002,Avon Books,http://images.amazon.com/images/P/0060002050.0...,http://images.amazon.com/images/P/0060002050.0...,http://images.amazon.com/images/P/0060002050.0...


In [47]:
books.head()

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...


In [48]:
# counting the number of ratings for each book. It is a series with book titles as index and number of ratings as values
number_of_ratings = ratings_with_books.groupby('Book-Title')['book_rating'].count()

In [49]:
number_of_ratings

Book-Title
 A Light in the Storm: The Civil War Diary of Amelia Martin, Fenwick Island, Delaware, 1861 (Dear America)    2
 Always Have Popsicles                                                                                        1
 Apple Magic (The Collector's series)                                                                         1
 Beyond IBM: Leadership Marketing and Finance for the 1990s                                                   1
 Clifford Visita El Hospital (Clifford El Gran Perro Colorado)                                                1
                                                                                                             ..
Ã?Ã?ber die Pflicht zum Ungehorsam gegen den Staat.                                                         3
Ã?Ã?lpiraten.                                                                                               1
Ã?Ã?rger mit Produkt X. Roman.                                                             

In [50]:
# resetting the index of the series to convert it back to a dataframe
number_of_ratings = number_of_ratings.reset_index()
number_of_ratings.head()

,Book-Title,book_rating
0,A Light in the Storm: The Civil War Diary of ...,2
1,Always Have Popsicles,1
2,Apple Magic (The Collector's series),1
3,Beyond IBM: Leadership Marketing and Finance ...,1
4,Clifford Visita El Hospital (Clifford El Gran...,1


In [51]:
number_of_ratings.rename(columns={'book_rating': 'number_of_ratings'}, inplace=True)

In [52]:
number_of_ratings.head(2)
number_of_ratings.shape

(160269, 2)

In [54]:
# calculating the average rating for each book. It is a series with book titles as index and average rating as values
avg_ratings = ratings_with_books.groupby('Book-Title')['book_rating'].mean().reset_index()
avg_ratings.rename(columns={'book_rating': 'avg_rating'}, inplace=True)

In [55]:
avg_ratings.head(2)

,Book-Title,avg_rating
0,A Light in the Storm: The Civil War Diary of ...,4.5
1,Always Have Popsicles,0.0


In [57]:
final_ratings = ratings_with_books.merge(number_of_ratings, on='Book-Title')

In [58]:
final_ratings.shape

(487671, 11)

In [59]:
final_ratings.head()

,user_id,ISBN,book_rating,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L,number_of_ratings
0,277427,002542730X,10,Politically Correct Bedtime Stories: Modern Ta...,James Finn Garner,1994,John Wiley &amp; Sons Inc,http://images.amazon.com/images/P/002542730X.0...,http://images.amazon.com/images/P/002542730X.0...,http://images.amazon.com/images/P/002542730X.0...,82
1,277427,0026217457,0,Vegetarian Times Complete Cookbook,Lucy Moll,1995,John Wiley &amp; Sons,http://images.amazon.com/images/P/0026217457.0...,http://images.amazon.com/images/P/0026217457.0...,http://images.amazon.com/images/P/0026217457.0...,7
2,277427,003008685X,8,Pioneers,James Fenimore Cooper,1974,Thomson Learning,http://images.amazon.com/images/P/003008685X.0...,http://images.amazon.com/images/P/003008685X.0...,http://images.amazon.com/images/P/003008685X.0...,1
3,277427,0030615321,0,"Ask for May, Settle for June (A Doonesbury book)",G. B. Trudeau,1982,Henry Holt &amp; Co,http://images.amazon.com/images/P/0030615321.0...,http://images.amazon.com/images/P/0030615321.0...,http://images.amazon.com/images/P/0030615321.0...,1
4,277427,0060002050,0,On a Wicked Dawn (Cynster Novels),Stephanie Laurens,2002,Avon Books,http://images.amazon.com/images/P/0060002050.0...,http://images.amazon.com/images/P/0060002050.0...,http://images.amazon.com/images/P/0060002050.0...,13


In [60]:
# keeping only those books which have more than 50 ratings
final_ratings = final_ratings[final_ratings['number_of_ratings'] > 50]

In [61]:
final_ratings.shape

(59903, 11)

In [62]:
# check for duplicates in final_ratings 
final_ratings.duplicated().any() # there are no duplicates

np.False_

In [64]:
# creating a pivot table with book titles as index, user ids as columns and book ratings as values
book_pivot = final_ratings.pivot_table(columns='user_id', index='Book-Title', values='book_rating')

In [65]:
book_pivot.shape

(703, 888)

### There are 703 books(reviewed more than 50 times by users) and 888 users(who reviewed more than 200 books)

In [66]:
book_pivot.head()

user_id,254,2276,2766,2977,3363,3757,4017,4385,6242,6251,...,274004,274061,274301,274308,274808,275970,277427,277478,277639,278418
Book-Title,,,,,,,,,,,,,,,,,,,,,
1984,9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN
1st to Die: A Novel,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2nd Chance,NaN,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN
4 Blondes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
84 Charing Cross Road,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,10.0,NaN,NaN,NaN,NaN


In [67]:
book_pivot.fillna(0, inplace=True)

user_id,254,2276,2766,2977,3363,3757,4017,4385,6242,6251,...,274004,274061,274301,274308,274808,275970,277427,277478,277639,278418
Book-Title,,,,,,,,,,,,,,,,,,,,,
1984,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1st to Die: A Novel,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2nd Chance,0.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4 Blondes,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
84 Charing Cross Road,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,10.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Year of Wonders,0.0,0.0,0.0,7.0,0.0,0.0,0.0,0.0,7.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
You Belong To Me,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Zen and the Art of Motorcycle Maintenance: An Inquiry into Values,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [68]:
book_pivot.head(2)

user_id,254,2276,2766,2977,3363,3757,4017,4385,6242,6251,...,274004,274061,274301,274308,274808,275970,277427,277478,277639,278418
Book-Title,,,,,,,,,,,,,,,,,,,,,
1984,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1st to Die: A Novel,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [69]:
from scipy.sparse import  csr_matrix
book_sparse = csr_matrix(book_pivot) # converting the pivot table to a sparse matrix for efficient storage and computation
book_sparse.shape

(703, 888)

### Trial 1: Using KNN algorithm

In [70]:
from sklearn.neighbors import NearestNeighbors
model = NearestNeighbors(algorithm='brute')

In [71]:
model.fit(book_sparse)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [72]:
# finding the 5 nearest neighbors for the first book in the pivot table
# values.reshape(1, -1)

#     Original: 1D array → shape like (features,)

#     After reshape: 2D array → shape (1, features)
# distances are distance between books in hyperspace & suggestions are the indices of the nearest neighbors
distances, suggestions = model.kneighbors(book_pivot.iloc[224,:].values.reshape(1,-1), n_neighbors=6) 

In [73]:
distances

array([[ 0.        , 67.73129098, 67.77802823, 72.22091879, 76.03909813,
        76.55027397]])

In [74]:
suggestions

array([[224, 227, 225, 228, 173, 277]])

In [75]:
for i in range(len(suggestions)):
    print(book_pivot.index[suggestions[i]])

Index(['Harry Potter and the Chamber of Secrets (Book 2)',
       'Harry Potter and the Prisoner of Azkaban (Book 3)',
       'Harry Potter and the Goblet of Fire (Book 4)',
       'Harry Potter and the Sorcerer's Stone (Book 1)', 'Exclusive',
       'Jacob Have I Loved'],
      dtype='str', name='Book-Title')


In [76]:
book_pivot.index[1]

'1st to Die: A Novel'

In [77]:
'Harry Potter and the Chamber of Secrets (Book 2)' in book_pivot.index

True

In [78]:
book_pivot.index.get_loc('Animal Farm')


47

In [79]:
def recommend_books(book_name):
    book_index = book_pivot.index.get_loc(book_name)
    distances, suggestions = model.kneighbors(book_pivot.iloc[book_index,:].values.reshape(1,-1), n_neighbors=6) 
    for i in range(len(suggestions[0])):
        if book_pivot.index[suggestions[0][i]] != book_name:
            print(book_pivot.index[suggestions[0][i]])

In [80]:
recommend_books('Animal Farm')

Exclusive
Jacob Have I Loved
Pleading Guilty
No Safe Place
Winter Moon


## Trail 2: Using cosine similarity 

In [81]:
from sklearn.metrics.pairwise import cosine_similarity

In [82]:
# calculating the cosine similarity between the books based on their ratings by users
similarity_score = cosine_similarity(book_pivot)
similarity_score.shape

(703, 703)

In [83]:
book_pivot.index[0] # The first book in the pivot table is '1984' and its index is 0

'1984'

In [84]:
# the similarity scores for the first book with all other books
similarity_score[0]

array([1.        , 0.09259251, 0.01122267, 0.        , 0.08708674,
       0.04847394, 0.02550819, 0.0730769 , 0.11207963, 0.02998295,
       0.03371421, 0.02134876, 0.06139476, 0.01868845, 0.0825378 ,
       0.06731328, 0.10252858, 0.04630962, 0.02314465, 0.10542489,
       0.        , 0.12738003, 0.0711696 , 0.05634353, 0.07792927,
       0.        , 0.06328801, 0.12354715, 0.06986576, 0.11043489,
       0.05842928, 0.01354254, 0.        , 0.07322549, 0.04157401,
       0.01419341, 0.0872911 , 0.01630857, 0.02242419, 0.07048157,
       0.10736773, 0.04839326, 0.07380827, 0.07787765, 0.07906263,
       0.0497226 , 0.05042114, 0.28393467, 0.08989427, 0.09244761,
       0.1152654 , 0.10803669, 0.0697622 , 0.04006742, 0.01765755,
       0.        , 0.05028458, 0.00501224, 0.07003198, 0.04718206,
       0.1738128 , 0.        , 0.01140469, 0.02743558, 0.03767901,
       0.11453863, 0.14358757, 0.        , 0.07197121, 0.12011177,
       0.06081257, 0.        , 0.        , 0.        , 0.09810

In [ ]:
# Recommendation function which takes the book name as input and returns the top 5 similar books based on the cosine similarity scores
def recommend_books(book_name):
    book_index = book_pivot.index.get_loc(book_name)
    similarity_scores = list(enumerate(similarity_score[book_index]))
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)
    similarity_scores = similarity_scores[1:6]
    book_indices = [i[0] for i in similarity_scores]
    data = []
    for i in book_indices:
        item = []
        temp_df = books[books['Book-Title'] == book_pivot.index[i]]
        item.extend(list(temp_df.drop_duplicates('Book-Title')['Book-Title'].values))
        item.extend(list(temp_df.drop_duplicates('Book-Title')['Book-Author'].values))
        item.extend(list(temp_df.drop_duplicates('Book-Title')['Image-URL-M'].values))
        data.append(item)
    return data


In [88]:
recommend_books('1984')

[['Animal Farm',
  'George Orwell',
  'http://images.amazon.com/images/P/0451526341.01.MZZZZZZZ.jpg'],
 ["The Handmaid's Tale",
  'Margaret Atwood',
  'http://images.amazon.com/images/P/0449212602.01.MZZZZZZZ.jpg'],
 ['The Catcher in the Rye',
  'J.D. Salinger',
  'http://images.amazon.com/images/P/0316769487.01.MZZZZZZZ.jpg'],
 ['Lord of the Flies',
  'William Gerald Golding',
  'http://images.amazon.com/images/P/0399501487.01.MZZZZZZZ.jpg'],
 ['Brave New World',
  'Aldous Huxley',
  'http://images.amazon.com/images/P/0060809833.01.MZZZZZZZ.jpg']]

In [89]:
import pickle

pickle.dump(book_pivot, open('book_pivot.pkl', 'wb'))
pickle.dump(similarity_score, open('similarity_score.pkl', 'wb'))
pickle.dump(books, open('books.pkl', 'wb'))